In [ ]:
import pandas as pd
import numpy as np

# --- scikit-learn ---
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
df = pd.read_excel(
    r"../Datos/Limpios/información_préstamos_limpio.xlsx")

# Eliminamos filas sin Prima (target)
df = df.dropna(subset=["Prima"]).copy()

In [41]:
## target y variables 

target = "Prima"

# columnas que no queremos usar como predictoras
cols_to_exclude = ["ID", "Prima", "Impago"]

feature_cols = [c for c in df.columns if c not in cols_to_exclude]

X = df[feature_cols]
y = df[target]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numeric_features = [c for c in numeric_features if c in X.columns]
categorical_features = [c for c in categorical_features if c in X.columns]

In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [42]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [43]:
def calcular_metricas(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    y_true_arr = np.array(y_true)
    y_pred_arr = np.array(y_pred)

    # Evita división por 0 por seguridad
    eps = 1e-8
    mape = np.mean(np.abs((y_true_arr - y_pred_arr) / np.maximum(np.abs(y_true_arr), eps))) * 100

    return rmse, mae, r2, mape


In [44]:
## MODELO XGBOOST con Gridsearch

from xgboost import XGBRegressor

xgb_reg = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_estimators=200,
    n_jobs=-1
)

xgb_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", xgb_reg),
    ]
)

param_grid_xgb = {
    "model__n_estimators": [200, 400],          # número de árboles
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
    "model__gamma": [0, 0.1, 0.5],
    "model__reg_lambda": [1, 5, 10],
}

xgb_grid = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid_xgb,
    scoring="neg_root_mean_squared_error",  # minimizamos RMSE
    cv=5,
    n_jobs=-1,
    verbose=2
)

print("Entrenando XGBoost con GridSearchCV...")
xgb_grid.fit(X_train, y_train)

print("\nMejores hiperparámetros encontrados para XGBoost:")
print(xgb_grid.best_params_)

best_xgb = xgb_grid.best_estimator_

y_pred_xgb = best_xgb.predict(X_test)
rmse_xgb, mae_xgb, r2_xgb, mape_xgb = calcular_metricas(y_test, y_pred_xgb)

print("\nResultados XGBoost en test:")
print(f"  RMSE: {rmse_xgb:.4f}")
print(f"  MAE : {mae_xgb:.4f}")
print(f"  R²  : {r2_xgb:.4f}")
print(f"  MAPE: {mape_xgb:.2f}%")

Entrenando XGBoost con GridSearchCV...
Fitting 5 folds for each of 1458 candidates, totalling 7290 fits

Mejores hiperparámetros encontrados para XGBoost:
{'model__colsample_bytree': 1.0, 'model__gamma': 0, 'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 400, 'model__reg_lambda': 1, 'model__subsample': 0.6}

Resultados XGBoost en test:
  RMSE: 25.9236
  MAE : 20.7565
  R²  : 0.9683
  MAPE: 3.74%


In [45]:
##RANDOM FOREST REGRESSOR

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ----------------------------------------------------------------------
# Random Forest Regressor + GridSearchCV
# ----------------------------------------------------------------------

# 1. Definimos el modelo base de Random Forest
rf_reg = RandomForestRegressor(
    random_state=42,
    n_jobs=-1,      # para usar todos los núcleos disponibles
)

# 2. Pipeline: preprocesamiento + modelo
rf_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),  # el ColumnTransformer que ya tienes
        ("model", rf_reg),
    ]
)

# 3. Grid de hiperparámetros para Random Forest
#    (hiperparámetros que habéis visto en clase: n_estimators, max_depth, max_features)
param_grid_rf = {
    "model__n_estimators": [100, 200, 300],    # nº de árboles en el bosque
    "model__max_depth": [None, 5, 10],         # profundidad máxima de cada árbol
    "model__max_features": ["sqrt", "log2"],   # nº de variables consideradas en cada split
}

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid_rf,
    scoring="neg_root_mean_squared_error",  # minimizamos RMSE
    cv=5,
    n_jobs=-1,
    verbose=2
)

print("Entrenando Random Forest con GridSearchCV...")
rf_grid.fit(X_train, y_train)

print("\nMejores hiperparámetros encontrados para Random Forest:")
print(rf_grid.best_params_)

best_rf = rf_grid.best_estimator_

# ----------------------------------------------------------------------
# Evaluación en test
# ----------------------------------------------------------------------

y_pred_rf = best_rf.predict(X_test)
rmse_rf, mae_rf, r2_rf, mape_rf = calcular_metricas(y_test, y_pred_rf)

print("\nResultados Random Forest en test:")
print(f"  RMSE: {rmse_rf:.4f}")
print(f"  MAE : {mae_rf:.4f}")
print(f"  R²  : {r2_rf:.4f}")
print(f"  MAPE: {mape_rf:.2f}%")



Entrenando Random Forest con GridSearchCV...
Fitting 5 folds for each of 18 candidates, totalling 90 fits

Mejores hiperparámetros encontrados para Random Forest:
{'model__max_depth': None, 'model__max_features': 'sqrt', 'model__n_estimators': 300}

Resultados Random Forest en test:
  RMSE: 68.3919
  MAE : 56.3573
  R²  : 0.7796
  MAPE: 10.68%


In [46]:
## ARBOL DE REGRESIÓN

from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ----------------------------------------------------------------------
# Árbol de Decisión para REGRESIÓN + GridSearchCV
# ----------------------------------------------------------------------

# 1. Definimos el modelo base de árbol
tree_reg = DecisionTreeRegressor(
    random_state=42
)

# 2. Pipeline: preprocesamiento + modelo
tree_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),  # el ColumnTransformer que ya tienes definido
        ("model", tree_reg),
    ]
)

# 3. Grid de hiperparámetros para el árbol (con lo visto en clase)
param_grid_tree = {
    "model__max_depth": [None, 3, 5, 7, 10],     # profundidad máxima del árbol
    "model__min_samples_split": [2, 5, 10, 20],  # mínimo de muestras para hacer un split
    "model__min_samples_leaf": [1, 2, 4, 8],     # mínimo de muestras en una hoja
}

tree_grid = GridSearchCV(
    estimator=tree_pipeline,
    param_grid=param_grid_tree,
    scoring="neg_root_mean_squared_error",  # optimizamos RMSE (en negativo)
    cv=5,
    n_jobs=-1,
    verbose=2
)

print("Entrenando Árbol de Regresión con GridSearchCV...")
tree_grid.fit(X_train, y_train)

print("\nMejores hiperparámetros encontrados para el Árbol de Regresión:")
print(tree_grid.best_params_)

best_tree = tree_grid.best_estimator_

# ----------------------------------------------------------------------
# Evaluación en test
# ----------------------------------------------------------------------

y_pred_tree = best_tree.predict(X_test)
rmse_tree, mae_tree, r2_tree, mape_tree = calcular_metricas(y_test, y_pred_tree)

print("\nResultados Árbol de Regresión en test:")
print(f"  RMSE: {rmse_tree:.4f}")
print(f"  MAE : {mae_tree:.4f}")
print(f"  R²  : {r2_tree:.4f}")
print(f"  MAPE: {mape_tree:.2f}%")



Entrenando Árbol de Regresión con GridSearchCV...
Fitting 5 folds for each of 80 candidates, totalling 400 fits

Mejores hiperparámetros encontrados para el Árbol de Regresión:
{'model__max_depth': None, 'model__min_samples_leaf': 8, 'model__min_samples_split': 20}

Resultados Árbol de Regresión en test:
  RMSE: 66.5120
  MAE : 51.7547
  R²  : 0.7916
  MAPE: 9.28%


In [47]:
## REGRESION LINEAL 
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ----------------------------------------------------------------------
# Preprocesamiento específico para regresión lineal
# (aquí SÍ escalamos)
# ----------------------------------------------------------------------

numeric_transformer_lr = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer_lr = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor_lr = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_lr, numeric_features),
        ("cat", categorical_transformer_lr, categorical_features),
    ]
)

# ----------------------------------------------------------------------
# Modelo: Regresión Lineal (Ridge para evitar multicolinealidad)
# ----------------------------------------------------------------------

ridge_reg = Ridge()

ridge_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor_lr),
        ("model", ridge_reg),
    ]
)

# Grid de hiperparámetros (muy sencillo, como en clase)
param_grid_ridge = {
    "model__alpha": [0.1, 1.0, 10.0, 50.0]
}

ridge_grid = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid_ridge,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    verbose=2
)

print("Entrenando Regresión Lineal (Ridge) con GridSearchCV...")
ridge_grid.fit(X_train, y_train)

print("\nMejores hiperparámetros para Regresión Lineal (Ridge):")
print(ridge_grid.best_params_)

best_ridge = ridge_grid.best_estimator_

# ----------------------------------------------------------------------
# Evaluación en test
# ----------------------------------------------------------------------

y_pred_ridge = best_ridge.predict(X_test)
rmse_ridge, mae_ridge, r2_ridge, mape_ridge = calcular_metricas(y_test, y_pred_ridge)

print("\nResultados Ridge en test:")
print(f"  RMSE: {rmse_ridge:.4f}")
print(f"  MAE : {mae_ridge:.4f}")
print(f"  R²  : {r2_ridge:.4f}")
print(f"  MAPE: {mape_ridge:.2f}%")



Entrenando Regresión Lineal (Ridge) con GridSearchCV...
Fitting 5 folds for each of 4 candidates, totalling 20 fits

Mejores hiperparámetros para Regresión Lineal (Ridge):
{'model__alpha': 1.0}

Resultados Ridge en test:
  RMSE: 73.0399
  MAE : 58.9141
  R²  : 0.7486
  MAPE: 10.61%


In [48]:
## LASSO

from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# ----------------------------------------------------------------------
# Preprocesamiento para regresión lineal (igual que Ridge)
# ----------------------------------------------------------------------

numeric_transformer_lr = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer_lr = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor_lr = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_lr, numeric_features),
        ("cat", categorical_transformer_lr, categorical_features),
    ]
)

# ----------------------------------------------------------------------
# Modelo: LASSO
# ----------------------------------------------------------------------

lasso_reg = Lasso(
    max_iter=10000,   # importante para que converja
    random_state=42
)

lasso_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor_lr),
        ("model", lasso_reg),
    ]
)

# Grid de hiperparámetros (alpha = fuerza de regularización)
param_grid_lasso = {
    "model__alpha": [0.001, 0.01, 0.1, 1.0, 10.0]
}

lasso_grid = GridSearchCV(
    estimator=lasso_pipeline,
    param_grid=param_grid_lasso,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    verbose=2
)

print("Entrenando LASSO con GridSearchCV...")
lasso_grid.fit(X_train, y_train)

print("\nMejores hiperparámetros para LASSO:")
print(lasso_grid.best_params_)

best_lasso = lasso_grid.best_estimator_

# ----------------------------------------------------------------------
# Evaluación en test
# ----------------------------------------------------------------------

y_pred_lasso = best_lasso.predict(X_test)
rmse_lasso, mae_lasso, r2_lasso, mape_lasso = calcular_metricas(y_test, y_pred_lasso)

print("\nResultados LASSO en test:")
print(f"  RMSE: {rmse_lasso:.4f}")
print(f"  MAE : {mae_lasso:.4f}")
print(f"  R²  : {r2_lasso:.4f}")
print(f"  MAPE: {mape_lasso:.2f}%")



Entrenando LASSO con GridSearchCV...
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Mejores hiperparámetros para LASSO:
{'model__alpha': 0.01}

Resultados LASSO en test:
  RMSE: 73.0416
  MAE : 58.9104
  R²  : 0.7486
  MAPE: 10.61%


In [49]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# ----------------------------------------------------------------------
# Modelo: Regresión Lineal Ridge
# ----------------------------------------------------------------------

ridge_reg = Ridge(
    random_state=42
)

ridge_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor_lr),
        ("model", ridge_reg),
    ]
)

# Grid de hiperparámetros para Ridge (fuerza de regularización)
param_grid_ridge = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0, 50.0]
}

ridge_grid = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid_ridge,
    scoring="neg_root_mean_squared_error",  # optimizamos RMSE (en negativo)
    cv=5,
    n_jobs=-1,
    verbose=2
)

print("Entrenando Regresión Ridge con GridSearchCV...")
ridge_grid.fit(X_train, y_train)

print("\nMejores hiperparámetros para Ridge:")
print(ridge_grid.best_params_)

best_ridge = ridge_grid.best_estimator_

# ----------------------------------------------------------------------
# Evaluación en test
# ----------------------------------------------------------------------

y_pred_ridge = best_ridge.predict(X_test)

rmse_ridge, mae_ridge, r2_ridge, mape_ridge = calcular_metricas(y_test, y_pred_ridge)

print("\nResultados Ridge en test:")
print(f"  RMSE: {rmse_ridge:.4f}")
print(f"  MAE : {mae_ridge:.4f}")
print(f"  R²  : {r2_ridge:.4f}")
print(f"  MAPE: {mape_ridge:.2f}%")


Entrenando Regresión Ridge con GridSearchCV...
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Mejores hiperparámetros para Ridge:
{'model__alpha': 1.0}

Resultados Ridge en test:
  RMSE: 73.0399
  MAE : 58.9141
  R²  : 0.7486
  MAPE: 10.61%


In [56]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# ----------------------------------------------------------------------
# LightGBM Regressor + GridSearchCV
# ----------------------------------------------------------------------

# 1. Modelo base LightGBM
lgbm_reg = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_estimators=200,
    n_jobs=-1
)

# 2. Pipeline: preprocesamiento + modelo
lgbm_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),  # mismo ColumnTransformer que usas con árboles/RF/XGBoost
        ("model", lgbm_reg),
    ]
)

# 3. Grid de hiperparámetros LightGBM
param_grid_lgbm = {
    "model__n_estimators": [200, 400],
    "model__num_leaves": [31, 63, 127],
    "model__max_depth": [-1, 5, 10],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
    "model__reg_lambda": [0.0, 1.0, 5.0],
}

lgbm_grid = GridSearchCV(
    estimator=lgbm_pipeline,
    param_grid=param_grid_lgbm,
    scoring="neg_root_mean_squared_error",  # minimizamos RMSE
    cv=5,
    n_jobs=-1,
    verbose=2
)

print("Entrenando LightGBM con GridSearchCV...")
lgbm_grid.fit(X_train, y_train)

print("\nMejores hiperparámetros encontrados para LightGBM:")
print(lgbm_grid.best_params_)

best_lgbm = lgbm_grid.best_estimator_

# ----------------------------------------------------------------------
# Evaluación en test
# ----------------------------------------------------------------------

y_pred_lgbm = best_lgbm.predict(X_test)
rmse_lgbm, mae_lgbm, r2_lgbm, mape_lgbm = calcular_metricas(y_test, y_pred_lgbm)

print("\nResultados LightGBM en test:")
print(f"  RMSE: {rmse_lgbm:.4f}")
print(f"  MAE : {mae_lgbm:.4f}")
print(f"  R²  : {r2_lgbm:.4f}")
print(f"  MAPE: {mape_lgbm:.2f}%")


Entrenando LightGBM con GridSearchCV...
Fitting 5 folds for each of 1458 candidates, totalling 7290 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000779 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1659
[LightGBM] [Info] Number of data points in the train set: 5311, number of used features: 25
[LightGBM] [Info] Start training from score 590.896198
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fur

c:\Users\ander\anaconda3\envs\RETO_07_MORADO\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [57]:
resultados = pd.DataFrame({
    "Modelo": ["Árbol", "RandomForest", "XGBoost", "LightGBM", "Ridge", "Lasso"],
    "RMSE": [rmse_tree, rmse_rf, rmse_xgb, rmse_lgbm, rmse_ridge, rmse_lasso],
    "MAE":  [mae_tree,  mae_rf,  mae_xgb,  mae_lgbm,  mae_ridge,  mae_lasso],
    "R2":   [r2_tree,   r2_rf,   r2_xgb,   r2_lgbm,   r2_ridge,   r2_lasso],
    "MAPE (%)": [mape_tree, mape_rf, mape_xgb, mape_lgbm, mape_ridge, mape_lasso]
})

resultados.sort_values("RMSE")


,Modelo,RMSE,MAE,R2,MAPE (%)
2,XGBoost,25.923587,20.756538,0.968336,3.742727
3,LightGBM,27.148522,20.926362,0.965273,3.663040
0,Árbol,66.512011,51.754724,0.791563,9.278825
1,RandomForest,68.391865,56.357260,0.779614,10.683694
4,Ridge,73.039876,58.914079,0.748641,10.609495
5,Lasso,73.041599,58.910440,0.748629,10.608146


In [58]:
# Estimación del coste esperado del seguro
# ---------------------------------------------------------

df_test = X_test.copy()

df_test["Prima_real"] = y_test.values
df_test["Coste_Esperado_Seguro"] = best_xgb.predict(X_test)

df_test.head()

,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,Estudios,Tipo_Jornada_Laboral,Estado_Civil,Posesion_Hipoteca,Personas_Cargo,Proposito,Fiador,Duracion_anios,Ingresos_totales_prestamo,Prima_real,Coste_Esperado_Seguro
1652,20,16992,45593,711,19,3,22.35,60,0.47,Escolar,jornada completa,Soltero,0,0,Vivienda,1,5,84960,626.05,602.904724
5302,21,19574,46664,848,29,1,13.80,36,0.55,Escolar,autónomo,Soltero,1,0,Vivienda,1,3,58722,378.85,374.748352
2986,69,37244,40000,367,473,2,3.81,48,0.26,Grado Universitario,jornada completa,Casado,1,1,Vivienda,0,4,148976,514.37,517.011902
1545,34,26596,53155,787,147,3,9.69,24,0.55,Escolar,desempleado,Casado,0,0,Vivienda,1,2,53192,341.82,338.099579
6016,68,39808,46344,575,499,2,19.15,48,0.30,Escolar,desempleado,Divorciado,1,0,Vivienda,1,4,159232,766.52,756.672241
